# Woodpile photonic crystal with controlled rod-segment defects

Voxelized permittivity distribution of a woodpile (FCC) photonic crystal with positive / negative
rod-segment defects, following

> S. Aeby, G. J. Aubry, N. Muller, F. Scheffold, *Scattering From Controlled Defects in Woodpile Photonic
> Crystals*, Adv. Optical Mater. **9**, 2001699 (2021).

The output grid is saved as `.h5` and loaded into Tidy3D through `AutomationModule`, exactly like the
random-slab workflow (`20250903_create_h5_from_ends.ipynb`). The generator and the plotting helper live in
`woodpile_helpers.py` next to this notebook. The membership test (circular test in the unwarped space `z' = z/s`, global
z-scale `s = aspect_ratio`) is the one of `create_permittivity_grid_penlike`; since woodpile rods are axis-aligned it is
separable, so the grid is stamped from 2-D cross-section masks and the filling-fraction bisection never builds the 3-D grid
(`perfect_voxel_ff`) - the paper-size `70 x 70 x 8.5 um^3` sample at 50 nm voxels takes a few seconds.

**Geometry conventions**

* Box `(Lx, Ly, Lz)` centred at the origin, voxel centres `(i + 0.5) dx - L/2`, `float32` grid, background first.
* Layers of parallel rods stacked along **z** with spacing `h = dz/4`; `dz = sqrt(2) d` gives FCC. Layer `k` is centred
  at `z = -Lz/2 + h/2 + layer_offset + k h`; even layers run along x, odd layers along y, every second layer of one
  orientation is shifted by `d/2`. Only layers whose centre lies in the box are generated (paper: `Lz = 5 dz` -> 20 layers).
* Rods have an elliptical cross-section: short semi-axis `b = minor_radius` in-plane, long semi-axis `a = aspect_ratio * b`
  along z (paper: `b = 0.15 um`, `a = 0.42 um`, `a/b = 2.8`, `r = sqrt(ab) = 0.25 um`). Adjacent layers overlap since `a > h/2`.
* A **rod segment** is the piece of rod of length `d` between two consecutive crossings with the rods of the layer *below*
  (`segment_ref='below'`); the layer above crosses it at its midpoint. Segment boundaries therefore sit at `n d` or `n d + d/2`.
* A **defect** replaces one complete segment by one whose cross-section **area** is `(1 + kappa)` times the regular one
  (both semi-axes scale by `sqrt(1 + kappa)`); `kappa = -1` removes the segment. Defects are drawn uniformly at random,
  without replacement, from the complete segments of rods whose axis lies inside the box, subject to the non-overlap rule
  (no two defects on the same / adjacent segments of a rod, on overlapping segments of neighbouring rods of a layer, or on
  crossing segments of adjacent layers). Rods carrying defects are split into pieces before voxelization, so a thinner
  segment never inherits voxels of the regular rod.


In [1]:
import numpy as np
import sys, os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# AutomationModule lives in the root of the tidy3d project
sys.path.append(os.path.abspath(r'H:\codes\tidy3d'))
import AutomationModule as AM

# generator + plotting helpers live next to this notebook (woodpile_helpers.py)
from woodpile_helpers import create_woodpile_dist, tables_to_dict, show_slice
help(create_woodpile_dist)

Help on function create_woodpile_dist in module woodpile_helpers:

create_woodpile_dist(box_size, grid_size, d, dz=None, permittivity=2.3409, background_permittivity=1.0, minor_radius=None, filling_fraction=None, aspect_ratio=2.8, n_defects=0, defect_density=None, kappa=0.0, seed=None, layer_offset=0.0, progress_every=None, verbose=False, segment_ref='below', ff_tolerance=0.001, ff_max_iter=25, save_rods=False, dir_save='./Structures')
    Voxelized permittivity of a woodpile photonic crystal with controlled rod-segment defects
    (Aeby, Aubry, Muller, Scheffold, Adv. Optical Mater. 9, 2001699 (2021)).

    Geometry conventions
    --------------------
    * Box of size (Lx, Ly, Lz) centred at the origin, voxel centres at (i + 0.5)*dx - L/2.
    * Layers of parallel rods are stacked along z with spacing h = dz/4 (dz = stacking period,
      default sqrt(2)*d -> FCC).  Layer k is centred at z = -Lz/2 + h/2 + layer_offset + k*h;
      even layers run along x, odd layers along y, and eve

## Paper crystal (perfect)

`d = 1.2 um`, `dz = sqrt(2) d`, `b = 0.15 um`, `a/b = 2.8`, `n = 1.53` (IP-Dip), air background, `Lz = 5 dz` (20 layers).
The paper sample is `70 x 70 x 8.5 um^3`; here a small `8d x 8d` footprint at ~50 nm voxels for a quick run.

In [3]:
# --- Parameters (paper values) ---
d = 1.2                          # in-plane rod pitch [um] the centre-to-centre distance between neighbouring parallel rods within one layer of the woodpile
dz = np.sqrt(2) * d              # stacking period (4 layers), FCC
ASPECT_RATIO = 1               # a/b, long axis along z  (a = 0.42 um)
FF_WOODPILE = 0.40
N_ROD = 2.5                     # IP-Dip
PERM = N_ROD**2
BACKGROUND = 1.0                 # air
L = 10 * d                        # in-plane box size [um]
BOX_SIZE = (L, L, 5 * dz)        # 5 FCC unit cells along z = 20 layers
DX_TARGET = 0.03                 # target voxel size [um]
GRID_SIZE = tuple(int(round(Li / DX_TARGET)) for Li in BOX_SIZE)
print("box =", tuple(round(x, 4) for x in BOX_SIZE), " grid =", GRID_SIZE)

eps0, rods0, defects0, ff0, info0 = create_woodpile_dist(
    BOX_SIZE, GRID_SIZE, d, dz=dz,
    permittivity=PERM, background_permittivity=BACKGROUND,
    filling_fraction=FF_WOODPILE, aspect_ratio=ASPECT_RATIO,
    n_defects=0, 
    verbose=True, save_rods=True,dir_save="./Structures/crystal"
)
print(f"layers: {info0['n_layers']}, rods: {info0['n_rods']}, complete segments: {info0['n_segments']}")
print(f"voxel ff = {ff0:.4f}   analytic (no overlap) pi a b / (d h) = {info0['ff_analytic']:.4f} size = {BOX_SIZE} grid = {GRID_SIZE}")

box = (12.0, 12.0, 8.4853)  grid = (400, 400, 283)
[ff bisection] it  0: b = 0.30000  ff = 0.51324  (target 0.4)
[ff bisection] it  1: b = 0.15000  ff = 0.14028  (target 0.4)
[ff bisection] it  2: b = 0.22500  ff = 0.30619  (target 0.4)
[ff bisection] it  3: b = 0.26250  ff = 0.41407  (target 0.4)
[ff bisection] it  4: b = 0.24375  ff = 0.36201  (target 0.4)
[ff bisection] it  5: b = 0.25312  ff = 0.38165  (target 0.4)
[ff bisection] it  6: b = 0.25781  ff = 0.39962  (target 0.4)
[ff bisection] best b = 0.25781, ff = 0.39962, residual = -0.00038
[woodpile] 20 layers (h = 0.4243), 210 rods, 1814 complete segments, 0 defects (kappa = 0.0)
[woodpile] b = 0.2578, a = 0.2578  ->  ff(voxel) = 0.3996, ff(defect-free) = 0.3996, ff(analytic, no overlap) = 0.4101
layers: 20, rods: 210, complete segments: 1814
voxel ff = 0.3996   analytic (no overlap) pi a b / (d h) = 0.4101 size = (12.0, 12.0, 8.48528137423857) grid = (400, 400, 283)


## Crystals with defects

Positive defects (`kappa = +2.8`, the +280 % of the paper's Figure 1d) and negative defects (`kappa = -0.64`, Figure 1e)
at number density `rho = 0.1 um^-3` (paper range 0 to 0.24 um^-3); plus a missing-segment example (`kappa = -1`).

In [4]:
RHO = 0.1        # defect number density [um^-3]
SEED = 12345

eps_pos, rods_pos, defects_pos, ff_pos, info_pos = create_woodpile_dist(
    BOX_SIZE, GRID_SIZE, d, dz=dz, permittivity=PERM, background_permittivity=BACKGROUND,
    filling_fraction=FF_WOODPILE, aspect_ratio=ASPECT_RATIO,
    defect_density=RHO, kappa=+2.8, seed=SEED, verbose=True,save_rods=True,dir_save="./Structures/deffects_positive"
)


[ff bisection] it  0: b = 0.30000  ff = 0.51324  (target 0.4)
[ff bisection] it  1: b = 0.15000  ff = 0.14028  (target 0.4)
[ff bisection] it  2: b = 0.22500  ff = 0.30619  (target 0.4)
[ff bisection] it  3: b = 0.26250  ff = 0.41407  (target 0.4)
[ff bisection] it  4: b = 0.24375  ff = 0.36201  (target 0.4)
[ff bisection] it  5: b = 0.25312  ff = 0.38165  (target 0.4)
[ff bisection] it  6: b = 0.25781  ff = 0.39962  (target 0.4)
[ff bisection] best b = 0.25781, ff = 0.39962, residual = -0.00038
[woodpile] 20 layers (h = 0.4243), 210 rods, 1814 complete segments, 122 defects (kappa = 2.8)
[woodpile] b = 0.2578, a = 0.2578  ->  ff(voxel) = 0.4506, ff(defect-free) = 0.3996, ff(analytic, no overlap) = 0.4101


In [12]:
eps_neg, rods_neg, defects_neg, ff_neg, info_neg = create_woodpile_dist(
    BOX_SIZE, GRID_SIZE, d, dz=dz, permittivity=PERM, background_permittivity=BACKGROUND,
    filling_fraction=FF_WOODPILE, aspect_ratio=ASPECT_RATIO,
    defect_density=RHO, kappa=-0.64, seed=SEED, verbose=True,save_rods=True,dir_save="./Structures/deffects_negative"
)


[ff bisection] it  0: b = 0.30000  ff = 0.52190  (target 0.4)
[ff bisection] it  1: b = 0.15000  ff = 0.13725  (target 0.4)
[ff bisection] it  2: b = 0.22500  ff = 0.29830  (target 0.4)
[ff bisection] it  3: b = 0.26250  ff = 0.40025  (target 0.4)
[ff bisection] best b = 0.26250, ff = 0.40025, residual = +0.00025
[woodpile] 20 layers (h = 0.4243), 170 rods, 1132 complete segments, 78 defects (kappa = -0.64)
[woodpile] b = 0.2625, a = 0.2625  ->  ff(voxel) = 0.3860, ff(defect-free) = 0.4002, ff(analytic, no overlap) = 0.4252


In [13]:
eps_miss, rods_miss, defects_miss, ff_miss, info_miss = create_woodpile_dist(
    BOX_SIZE, GRID_SIZE, d, dz=dz, permittivity=PERM, background_permittivity=BACKGROUND,
    filling_fraction=FF_WOODPILE, aspect_ratio=ASPECT_RATIO,
    defect_density=RHO, kappa=-1.0, seed=SEED, verbose=True,save_rods=True,dir_save="./Structures/deffects_missing"
)


[ff bisection] it  0: b = 0.30000  ff = 0.52190  (target 0.4)
[ff bisection] it  1: b = 0.15000  ff = 0.13725  (target 0.4)
[ff bisection] it  2: b = 0.22500  ff = 0.29830  (target 0.4)
[ff bisection] it  3: b = 0.26250  ff = 0.40025  (target 0.4)
[ff bisection] best b = 0.26250, ff = 0.40025, residual = +0.00025
[woodpile] 20 layers (h = 0.4243), 170 rods, 1132 complete segments, 78 defects (kappa = -1.0)
[woodpile] b = 0.2625, a = 0.2625  ->  ff(voxel) = 0.3764, ff(defect-free) = 0.4002, ff(analytic, no overlap) = 0.4252


In [14]:
print()
print(f"{'structure':>12s} {'n_def':>6s} {'kappa':>7s} {'ff voxel':>9s} {'ff est.':>9s}")
for name, ffv, inf in [("perfect", ff0, info0), ("positive", ff_pos, info_pos),
                       ("negative", ff_neg, info_neg), ("missing", ff_miss, info_miss)]:
    print(f"{name:>12s} {inf['n_defects']:6d} {inf['kappa']:7.2f} {ffv:9.4f} {inf['ff_defect_estimate']:9.4f}")
print("defect table (first 3 rows):")
print(defects_pos[:3])


   structure  n_def   kappa  ff voxel   ff est.
     perfect      0    0.00    0.4002    0.4002
    positive     78    2.80    0.4526    0.4728
    negative     78   -0.64    0.3860    0.3837
     missing     78   -1.00    0.3764    0.3743
defect table (first 3 rows):
[(3. , -3. ,  3.60624458, 2.4, -3. ,  3.60624458, 3.6, -3. ,  3.60624458, 18, 'x', 155, 2, 2.8, 0.5117067, 0.5117067)
 (0. ,  1.8, -0.21213203, 0. ,  1.2, -0.21213203, 0. ,  2.4, -0.21213203,  9, 'y',  81, 1, 2.8, 0.5117067, 0.5117067)
 (4.2,  0.6,  1.90918831, 3.6,  0.6,  1.90918831, 4.8,  0.6,  1.90918831, 14, 'x', 124, 3, 2.8, 0.5117067, 0.5117067)]


## Sanity plots

Cross-sections of the perfect crystal: `xz` through the axis of an x-rod (x-rods appear as stripes, y-rods as ellipses
with the long axis along z), `yz` through the axis of a y-rod, and one `xy` slice per layer orientation.

In [ ]:
# an x-rod of layer 0 at y = 0 and a y-rod of layer 1 at x = 0
xrod = rods0[(rods0['layer'] == 0) & (np.isclose(rods0['position'], 0.0))][0]
yrod = rods0[(rods0['layer'] == 1) & (np.isclose(rods0['position'], 0.0))][0]

fig, axs = plt.subplots(2, 2, figsize=(12, 9))
show_slice(eps0, BOX_SIZE, 1, xrod['position'], axs[0, 0], title=f"xz slice through x-rod axis (y = {xrod['position']:.2f} um)")
show_slice(eps0, BOX_SIZE, 0, yrod['position'], axs[0, 1], title=f"yz slice through y-rod axis (x = {yrod['position']:.2f} um)")
show_slice(eps0, BOX_SIZE, 2, xrod['z'], axs[1, 0], title=f"xy slice, layer 0 (x-rods), z = {xrod['z']:.3f} um")
show_slice(eps0, BOX_SIZE, 2, yrod['z'], axs[1, 1], title=f"xy slice, layer 1 (y-rods), z = {yrod['z']:.3f} um")
for ax in axs[0]:
    for zk in info0['z_layers']:
        ax.axhline(zk, color='tab:blue', lw=0.4, alpha=0.5)
plt.tight_layout(); plt.show()

Slices through one defect segment of each type (red boxes = defect segment extents; the neighbouring
crossing rods of the adjacent layers are left intact).

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(12, 13))
for row, (name, eps_d, defects_d) in enumerate([("positive kappa=+2.8", eps_pos, defects_pos),
                                                 ("negative kappa=-0.64", eps_neg, defects_neg),
                                                 ("missing kappa=-1", eps_miss, defects_miss)]):
    df = defects_d[defects_d['orientation'] == 'x'][0]      # one x-oriented defect
    show_slice(eps_d, BOX_SIZE, 1, df['y'], axs[row, 0], defects=defects_d,
               title=f"{name}: xz slice through defect rod (y = {df['y']:.2f} um)")
    show_slice(eps_d, BOX_SIZE, 2, df['z'], axs[row, 1], defects=defects_d,
               title=f"{name}: xy slice at defect layer {df['layer']} (z = {df['z']:.3f} um)")
    axs[row, 0].set_xlim(df['x'] - 2 * d, df['x'] + 2 * d); axs[row, 0].set_ylim(df['z'] - 1.5 * dz / 2, df['z'] + 1.5 * dz / 2)
    axs[row, 1].set_xlim(df['x'] - 2 * d, df['x'] + 2 * d); axs[row, 1].set_ylim(df['y'] - 2 * d, df['y'] + 2 * d)
plt.tight_layout(); plt.show()

Defect positions: histograms of the segment centres along x, y, z and per layer (uniform placement over the volume).
Segment centres live on a discrete `d/2` grid in-plane and on the layer grid in z, so the bins are aligned to it.
The positive and negative structures use the same `seed`, hence identical defect positions (the histograms coincide).

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(16, 3.5))
bins_xy = np.arange(-L / 2, L / 2 + d / 2, d / 2) - d / 4
bins_z = np.concatenate([info0['z_layers'] - info0['layer_spacing'] / 2, [info0['z_layers'][-1] + info0['layer_spacing'] / 2]])
for ax, key, bins in zip(axs[:3], 'xyz', [bins_xy, bins_xy, bins_z]):
    ax.hist(defects_pos[key], bins=bins, alpha=0.6, label='positive')
    ax.hist(defects_neg[key], bins=bins, alpha=0.6, label='negative', histtype='step', lw=1.5)
    ax.set_xlabel(f"defect centre {key} [um]"); ax.set_ylabel("count")
axs[3].hist(defects_pos['layer'], bins=np.arange(info0['n_layers'] + 1) - 0.5, alpha=0.6, label='positive')
axs[3].hist(defects_neg['layer'], bins=np.arange(info0['n_layers'] + 1) - 0.5, alpha=0.6, label='negative', histtype='step', lw=1.5)
axs[3].set_xlabel("layer index"); axs[3].legend()
plt.tight_layout(); plt.show()

print(f"perfect crystal : voxel ff = {ff0:.4f} | analytic no-overlap ff = pi a b/(d h) = {info0['ff_analytic']:.4f} | paper ~0.35")
print(f"positive defects: voxel ff = {ff_pos:.4f} ({info_pos['n_defects']} defects, kappa = {info_pos['kappa']:+.2f})")
print(f"negative defects: voxel ff = {ff_neg:.4f} ({info_neg['n_defects']} defects, kappa = {info_neg['kappa']:+.2f})")
print(f"missing segments: voxel ff = {ff_miss:.4f} ({info_miss['n_defects']} defects, kappa = {info_miss['kappa']:+.2f})")

## Save

Permittivity grid with the usual `n_{n:.2f}_ff_{ff:.4f}.h5` naming (loadable through `AutomationModule`), plus a second
file with the `rods` and `defects` tables (orientation stored as `0 = x`, `1 = y`) so the defect positions are recoverable.